In [1]:
import pandas as pd
import numpy as np
import samplics
from samplics.estimation import TaylorEstimator, ReplicateEstimator
from samplics.utils import PopParam, SinglePSUEst

In [ ]:
path = r'data/LLCP2022.XPT'
df = pd.read_sas(path)

# Restrict data to South Dakota
sd = df[df['_STATE']==46].copy()

# Re-code smoker variable to binary and null for non response
sd['CURRENTUSE'] = sd['_RFSMOK3'].replace({1: 0, 
                                           2: 1, 
                                           9: np.nan})

# Prevalence and 95% CIs with PopParam.prop
stratified_prop_estimator = TaylorEstimator(PopParam.prop)
stratified_prop_estimator.estimate(
    y=sd["CURRENTUSE"],
    samp_weight=sd["_LLCPWT"],
    stratum=sd["_STSTR"],
    psu=sd["_PSU"],
    remove_nan=True)
prev_prop = stratified_prop_estimator.to_dataframe()
print(prev_prop)
print()

# Prevalence and 95% CIs with PopParam.mean
stratified_mean_estimator = TaylorEstimator(PopParam.mean)
stratified_mean_estimator.estimate(
    y=sd["CURRENTUSE"],
    samp_weight=sd["_LLCPWT"],
    stratum=sd["_STSTR"],
    psu=sd["_PSU"],
    remove_nan=True)
prev_mean = stratified_mean_estimator.to_dataframe()
print(prev_mean)

          _param  _level  _estimate  _stderror      _lci      _uci       _cv
0  PopParam.prop     0.0   0.859841   0.012199  0.834178  0.882093  0.014188
1  PopParam.prop     1.0   0.140159   0.012199  0.117907  0.165822  0.087040

          _param  _estimate  _stderror      _lci      _uci      _cv
0  PopParam.mean   0.140159   0.012199  0.116245  0.164074  0.08704


In [ ]:
# samplics (PopParam.prop) - LCI: 11.7907   UCI: 16.5822
# R (svyciprop)            - LCI: 11.7906   UCI: 16.5823
# SAS (proc survyefreq)    - LCI: 11.6244   UCI: 16.4075

# samplics (PopParam.mean) - LCI: 11.6245   UCI: 16.4074
# R (svymeans)             - LCI: 11.62477  UCI: 16.40706
# SAS (proc surveymeans)   - LCI: 11.624463 UCI: 16.407363